# Milestone 5: Ensembling

**Deadline:**  `Jul 20, 2026`

- Extracting and sorting logits to get the top 3 predictions (e.g., A C B).
- Ensemble techniques to stack predictions from multiple models.
- Other strategies to improve predictions.

<!--
gdown --folder 1QO1vigWXqH_FyHiOrXU94ogUlQivRzqj
 -->

## Import libraries

In [1]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## Config

In [2]:
DATA_DIR = Path("dataset/")
SUBMISSION_PATH = Path("submission.csv")

OPTIONS = ["A", "B", "C", "D", "E"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

## Helper Functions

In [3]:
def path_to_str(path: Path) -> str:
    return str(path.absolute())

In [4]:
def load_classifier(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

    return tokenizer, model.to(DEVICE).eval()

In [5]:
@torch.no_grad()
def predict_proba(tokenizer, model, texts):
    all_probs = []
    for i in range(0, len(texts), 32):
        batch = texts[i : i + 32]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
        ).to(DEVICE)

        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs.cpu())
    return torch.cat(all_probs, dim=0)

In [6]:
def format_prompt(row):
    parts = [row["prompt"]]
    for op in OPTIONS:
        parts.append(f"{op}) {row[op]}")

    return " ".join(parts)

In [7]:
def weighted_ensemble(deberta_probs, roberta_probs):
    return 0.7 * deberta_probs + 0.3 * roberta_probs

In [8]:
def top_k_predictions(probs, k=3):
    ranking = probs.argsort(dim=-1, descending=True)[:, :k].tolist()

    return [" ".join(OPTIONS[i] for i in row) for row in ranking]

In [9]:
def map_at_3(predictions, answers):
    score = 0.0

    for prediction, answer in zip(predictions, answers):
        ranked = prediction.split()
        score += 1 / (ranked.index(answer) + 1) if answer in ranked else 0.0

    return score / len(predictions)

## Setup

Use the following two fine-tuned sequence classification checkpoints:

**DeBERTa:** `microsoft/deberta-v3-small` (fine-tuned checkpoint) \
**RoBERTa:** `roberta-base` (fine-tuned checkpoint)

**Label Mapping:** The models output logits for five labels corresponding to the answer options:

```
Label ID         Option

0                   A
1                   B
2                   C
3                   D
4                   E
```

In [10]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": path_to_str(DATA_DIR / "train.csv"),
        "test": path_to_str(DATA_DIR / "test.csv"),
    },
)

In [11]:
train = dataset["train"]
test = dataset["test"]
assert isinstance(train, Dataset)
assert isinstance(test, Dataset)

In [12]:
deberta_tokenizer, deberta = load_classifier("microsoft/deberta-v3-small")
roberta_tokenizer, roberta = load_classifier("roberta-base")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.de

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Question 1:

Load the fine-tuned **DeBERTa** and **RoBERTa** models.

For the prompt at **row index 25**, perform inference using each model independently and apply **Softmax** to obtain class probabilities.

**Question 1:**

Which answer option receives the highest probability from the **DeBERTa** model, and what is that probability?

*(answer format : eg - A, probability of A)*

In [13]:
row_25 = train[25]
assert isinstance(row_25, dict)

prompt_25 = format_prompt(row_25)

deberta_probs_25 = predict_proba(deberta_tokenizer, deberta, prompt_25)[0]
roberta_probs_25 = predict_proba(roberta_tokenizer, roberta, prompt_25)[0]

deberta_probs_25

tensor([0.1862, 0.1752, 0.1921, 0.2512, 0.1951], dtype=torch.float16)

In [14]:
OPTIONS[int(deberta_probs_25.argmax())], round(deberta_probs_25.max().item(), 4)

('D', 0.2512)

### Question 2:

Using the same sample (**row index 25**), average the class probabilities from both models.

```
Average Probability = [P(DeBERTa) + P(RoBERTa)]/2
```

**Question 2:**

Which answer option receives the highest averaged probability after simple probability ensembling?

In [15]:
average_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2
average_probs_25

tensor([0.1977, 0.1721, 0.2007, 0.2106, 0.2187])

In [16]:
OPTIONS[int(average_probs_25.argmax())]

'E'

### Question 3:

Apply **weighted probability averaging after Softmax** using the following weights:

**DeBERTa:** `0.70`

**RoBERTa:** `0.30`

Compute:

```
P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]
```

**Question 3:**

Which answer option is ranked first after weighted ensembling?

In [17]:
weighted_probs_25 = weighted_ensemble(deberta_probs_25, roberta_probs_25)
weighted_probs_25

tensor([0.1932, 0.1733, 0.1973, 0.2269, 0.2093])

In [18]:
OPTIONS[int(weighted_probs_25.argmax())]

'D'

### Question 4:

Using the weighted ensemble probabilities from **Q3**, rank all five answer options.

Write the final prediction exactly in **Kaggle submission format**.

**Question 4:**

What is the Top-3 prediction string for row index 25?

**Example:** `C A E`

In [19]:
top_k_predictions(weighted_probs_25.unsqueeze(0))[0]

'D E C'

### Question 5:

Run the weighted ensemble pipeline on every row of `test.csv`.

Save the predictions in a file named `submission.csv` using the required Kaggle format:

```
id,prediction
```

where the **prediction** column contains the Top-3 ranked options separated by spaces.

**Question 5:**

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [20]:
test_prompts = [format_prompt(r) for r in test]

test_deberta_probs = predict_proba(deberta_tokenizer, deberta, test_prompts)
test_roberta_probs = predict_proba(roberta_tokenizer, roberta, test_prompts)
test_weighted_probs = weighted_ensemble(test_deberta_probs, test_roberta_probs)

test_weighted_probs.shape

torch.Size([500, 5])

In [21]:
submission = pd.DataFrame(
    {"id": test["id"], "prediction": top_k_predictions(test_weighted_probs)}
)

submission.to_csv(SUBMISSION_PATH, index=False)
submission.head()

,id,prediction
0,1,D E C
1,2,D E C
2,3,D E C
3,4,D E C
4,5,D E C


In [22]:
len(pd.read_csv(SUBMISSION_PATH))

500

### Question 6:

For the first **50 rows** of `test.csv`, create two versions of every prompt:

1. Original prompt
2. Instruction-augmented prompt by prepending: `"Answer the following multiple-choice question carefully:"`

Run inference using **DeBERTa** on both versions.

Average the predicted probabilities from both passes.

**Question 6:**

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [31]:
first_50_prompts = test_prompts[:50]
augmented_prompts = [
    f"Answer the following multiple-choice question carefully: {prompt}"
    for prompt in first_50_prompts
]  # noqa: E501

original_probs = predict_proba(deberta_tokenizer, deberta, first_50_prompts)
augmented_probs = predict_proba(deberta_tokenizer, deberta, augmented_prompts)
tta_probs = (original_probs + augmented_probs) / 2

In [32]:
int((original_probs.argmax(dim=-1) != tta_probs.argmax(dim=-1)).sum())

0

### Question 7:

Process the first **100 rows** of `test.csv`. And compare the Top-1 prediction from:
1. DeBERTa
2. Weighted Ensemble

**Question 7:**

How many rows have different Top-1 predictions?

In [33]:
deberta_probs_100 = test_deberta_probs[:100]
weighted_probs_100 = test_weighted_probs[:100]

In [34]:
int((deberta_probs_100.argmax(dim=-1) != weighted_probs_100.argmax(dim=-1)).sum())

0

### Question 8:

For the first **100 rows** of `test.csv`, record the highest class probability (confidence) predicted by:

1. DeBERTa
2. Weighted Ensemble

For every row, compute:

```
Confidence Gain = Ensemble Confidence−DeBERTa Confidence
```

**Question 8:**

How many rows have a positive confidence gain (greater than 0)?

In [35]:
confidence_gain = (
    weighted_probs_100.max(dim=-1).values - deberta_probs_100.max(dim=-1).values
)

int((confidence_gain > 0).sum())

0

### Question 9:

For the first **100 rows** of `test.csv`, compare the Top-3 prediction strings generated by:

1. DeBERTa alone
2. Weighted Ensemble

**Question 9:**

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

**Examples:** `A C D vs. A D C`

In [36]:
deberta_top_3 = top_k_predictions(deberta_probs_100)
weighted_top_3 = top_k_predictions(weighted_probs_100)

sum(
    deberta_prediction != weighted_prediction
    for deberta_prediction, weighted_prediction in zip(deberta_top_3, weighted_top_3)
)

0

### Question 10:

Using the Top-3 predictions generated by your weighted ensemble for the **first 100 validation samples**, compute the **MAP@3** score.

**Question 10:**

What is the final **MAP@3 score**?

*(Round to 4 decimal places.)*

In [37]:
validation = train[:100]
assert isinstance(validation, dict)

validation_deberta_probs = predict_proba(
    deberta_tokenizer, deberta, validation["prompt"]
)
validation_roberta_probs = predict_proba(
    roberta_tokenizer, roberta, validation["prompt"]
)
validation_weighted_probs = weighted_ensemble(
    validation_deberta_probs, validation_roberta_probs
)

In [38]:
round(map_at_3(top_k_predictions(validation_weighted_probs), validation["answer"]), 4)

0.2967